# JW $Z_4^{TF}$ system - cocycle extraction

Created: 07-09-2026

Objectives:
* Iterate on [this notebook](jw_z_4_tf_system_improved_disentanglers.ipynb), now extract $\nu_2, \omega_2$ (depending on your convention) and extract associated invariants, coboundaries, etc.

# Imports

In [1]:
import numpy as np

In [2]:
import jax
jax.config.update('jax_platform_name', 'cpu')

import jax.numpy as jnp

In [3]:
import matplotlib.pyplot as plt

In [4]:
from tqdm import tqdm

In [5]:
from functools import reduce
from operator import mul
from itertools import product

In [6]:
from random import random

In [7]:
import quimb.tensor as qtn
import quimb as qu

In [8]:
from scipy.stats import unitary_group

In [9]:
from collections import Counter

In [10]:
import pandas as pd

In [11]:
from time import time

In [12]:
from humanize import naturalsize

# Definitions
## Construct cluster state

In [13]:
np_up_X_state = 1/(np.sqrt(2))*np.array([1,1])

In [14]:
qu_up_X_state = qtn.Tensor(
    data=np_up_X_state,
    inds=('k',),
    tags='prod'
)

In [15]:
np_CZ = np.diag([1,1,1,-1])

In [16]:
np_CZ = np_CZ.reshape((2,)*4)

In [17]:
qu_CZ = qtn.Tensor(
    data=np_CZ,
    inds=('k1', 'k2', 'b1', 'b2'),
    tags='CZ'
)

In [18]:
np_hadamard = np.pow(2, -1/2)*np.array([
    [1,1],
    [1,-1]
])

In [19]:
qu_hadamard = qtn.Tensor(data=np_hadamard, inds=('k', 'b'), tags='Had')

In [20]:
def get_cluster_state_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        qu_up_X_state.reindex({'k': f'kc_1_{i}'})
        for i in range(num_sites)
    ]

    first_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_1_{i}',
            'b2': f'kc_1_{i+1}',
            'k1': f'kc_2_{i}',
            'k2': f'kc_2_{i+1}'
        })
        for i in range(0, num_sites, 2)
    ]


    second_layer_circuit_tensors = [
        qu_CZ.reindex({
            'b1': f'kc_2_{i}',
            'b2': f'kc_2_{(i+1)%num_sites}',
            'k1': f'kh_{i}',
            'k2': f'k{(i+1)%num_sites}'
        })
        for i in range(1, num_sites+1, 2)
    ]

    hadamard_layer = [
        qu_hadamard.reindex({
            'b': f'kh_{i}',
            'k': f'k{i}'
        })
        for i in range(1, num_sites, 2)
    ]
    all_tensors = (
        product_state_tensors
        + first_layer_circuit_tensors
        + second_layer_circuit_tensors
        + hadamard_layer
    )

    out = qtn.TensorNetwork(all_tensors, virtual=True)
    out.mangle_inner_()

    return out

## Construct product state

In [21]:
np_up_X_state = 1/(np.sqrt(2))*np.array([1,1])

In [22]:
np_up_Z_state = np.array([1,0])

In [23]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='prod'
)

In [24]:
alternating_states = [
    qu_up_X_state,
    qu_up_Z_state
]

def get_product_qu_tensor_network(num_sites):
    assert (num_sites%2) == 0

    product_state_tensors = [
        alternating_states[i%2].reindex({'k': f'k{i}'})
        for i in range(num_sites)
    ]

    out = qtn.TensorNetwork(
        product_state_tensors,
        virtual=True
    )
    out.mangle_inner_()

    return out

## Symmetries

In [25]:
def multikron(arrays):
    return reduce(np.kron, arrays)

In [26]:
np_I = np.array([
    [1,0],
    [0,1]
])

np_X = np.array([
    [0,1],
    [1,0]
])

np_Y = np.array([
    [0,-1j],
    [1j,0]
])

np_Z = np.array([
    [1,0],
    [0,-1]
])

In [27]:
qu_I = qtn.Tensor(
    np_I,
    inds=['k', 'b'],
    tags='X'
)

qu_X = qtn.Tensor(
    np_X,
    inds=['k', 'b'],
    tags='X'
)

qu_Y = qtn.Tensor(
    np_Y,
    inds=['k', 'b'],
    tags='Y'
)

qu_Z = qtn.Tensor(
    np_Z,
    inds=['k', 'b'],
    tags='Z'
)

In [28]:
def get_multisite_qu_X(num_sites):
    np_many_X = multikron([np_X]*num_sites)

    out = qtn.Tensor(
        np_many_X,
        inds=['k', 'b'],
        tags='mulit_site_X',
    )

    return out

In [29]:
def get_multisite_qu_I(num_sites):
    np_many_I = multikron([np_I]*num_sites)

    out = qtn.Tensor(
        np_many_I,
        inds=['k', 'b'],
        tags='mulit_site_I',
    )

    return out

In [30]:
qu_spin_fermion_fp = (
    qu_I.reindex({'k': 'ks', 'b': 'bs'})
    & qu_Z.reindex({'k': 'kf', 'b': 'bf'})
).contract()

qu_unit_cell_fp = qu_spin_fermion_fp.fuse({
    'k': ['ks', 'kf'],
    'b': ['bs', 'bf']
})

In [31]:
"""
def get_multisite_qu_fp(num_sites):
    assert (num_sites%2)==0

    num_unit_cells = (num_sites//2)

    np
"""

'\ndef get_multisite_qu_fp(num_sites):\n    assert (num_sites%2)==0\n\n    num_unit_cells = (num_sites//2)\n\n    np\n'

Decompose T symmetry as $MK$:

In [32]:
np_00 = np.array([[1,0], [0,0]])
np_11 = np.array([[0,0], [0,1]])

In [33]:
def tensor_product_operators(op_1, op_2):
    out = (
        op_1[..., np.newaxis, np.newaxis]
        *op_2[np.newaxis, np.newaxis, ...]
    )

    return out

In [34]:
np_M = (
    tensor_product_operators(np_X, np_00)
    + tensor_product_operators(np_Y, np_11)
)

In [35]:
qu_M = qtn.Tensor(
    np_M,
    inds=['ks', 'bs', 'kf', 'bf']
)

In [36]:
np_M_reindexed = (
    qu_M
    .fuse({
        'k': ['ks', 'kf'],
        'b': ['bs', 'bf']
    })
    .transpose('k', 'b')
    .data
)

## Extracting projectors

In [37]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [38]:
def maximize_projector_states(rho, left_sites, proj_sites, right_sites):
    v = qtn.Tensor(
        data=random_uniform_complex((2,)*(len(proj_sites)-1)),
        inds=[f'k{i}' for i in proj_sites[:-1]]
    )

    tnopt = qtn.TNOptimizer(
        v,  # the tensor network we want to optimize
        loss_func,  # the function we want to minimize
        norm_fn=normalize_v,
        loss_constants={"rho": rho},
        loss_kwargs={
            "left_sites": left_sites,
            "proj_sites": proj_sites,
            "right_sites": right_sites,
        },
        autodiff_backend="jax",
        optimizer="L-BFGS-B",
        progbar=False
    )

    v_opt = tnopt.optimize(n=2000)

    embedded_v_opt = embed_fp_even_vector(v_opt).contract()

    return v_opt, embedded_v_opt, tnopt.losses

In [39]:
def projector_state_check(state, rho, left_sites):
    left_right_rho = (
        rho
        & state
        & state.conj().reindex({s: f'b{s[1:]}' for s in state.inds})
    )

    left_right_rho = left_right_rho.contract()

    left_inds = [
        f'{s}{i}'
        for i in left_sites
        for s in 'kb'
    ]

    schmidt_decomp = qtn.tensor_core.tensor_split(
        left_right_rho,
        left_inds=left_inds,
        method='svd',
        #cutoff=1e-6,
        cutoff_mode='abs',
        absorb=None,
        renorm=False,
        bond_ind='v'
    )

    schmidt_vals = schmidt_decomp.tensors[1]

    return schmidt_vals

### Embed vector

In [40]:
np_CX = (
    tensor_product_operators(np_00, np_I)
    + tensor_product_operators(np_11, np_X)
)

In [41]:
qu_CX = qtn.Tensor(
    np_CX,
    inds=['k1', 'b1', 'k2', 'b2']
)

In [42]:
np_up_Z_state = np.array([1,0])

In [43]:
qu_up_Z_state = qtn.Tensor(
    data=np_up_Z_state,
    inds=('k',),
    tags='Z0_pad'
)

In [131]:
def embed_fp_even_vector(v):
    # Take a vector of length 2N-1, and return a vector of length 2N
    # which commutes with IZIZ...IZIZ
    # Assuming v site ordering is spin-fermion-spin-...-fermion.

    sites = sorted(int(s[1:]) for s in v.inds)
    padded_site = sites[-1] + 1
    num_cx_gates = len(sites)//2
    
    if num_cx_gates>=1:
        padded_v = (
            v
            & qu_up_Z_state.reindex({'k': f'k{padded_site}_0'})
        )
    else:
        padded_v = (
            v
            & qu_up_Z_state.reindex({'k': f'k{padded_site}'})
        )
    num_cx_gates = len(sites)//2
    
    cx_gates = [
        qu_CX.reindex({
            'k1': f'k{sites[2*i+1]}',
            'b1': f'b{sites[2*i+1]}',
            'k2': f'k{padded_site}_{i+1}',
            'b2': f'k{padded_site}_{i}'
        })
        for i in range(num_cx_gates-1)
    ]
    
    # Handling annoying edge case logic
    if num_cx_gates >= 1:
        i = num_cx_gates-1
        cx_gates.append(
            qu_CX.reindex({
                'k1': f'k{sites[2*i+1]}',
                'b1': f'b{sites[2*i+1]}',
                'k2': f'k{padded_site}',
                'b2': f'k{padded_site}_{i}'
            })
        )
    
    reindexed_padded_v = (
        padded_v
        .reindex({
            f'k{sites[2*i+1]}': f'b{sites[2*i+1]}'
            for i in range(num_cx_gates)
        })
    )
    sym_v = qtn.TensorNetwork([
        reindexed_padded_v,
        *cx_gates
    ])
    
    sym_v.mangle_inner_()

    return sym_v

### Loss function

In [45]:
def get_rho_purity(rho, sites):
    # Assuming rho is a Hermitian reduced density matrix
    reindex_map = (
        {f'k{i}': f'b{i}' for i in sites}
        | {f'b{i}': f'k{i}' for i in sites}
    )

    rho_other = rho.reindex(reindex_map)
    #rho_other.mangle_inner_()
    
    out = (rho & rho_other).contract()

    return out

In [46]:
def normalize_v(v):
    norm = (v & v.conj()).contract()
    w = v*jnp.power(norm, -0.5)
    return w

In [47]:
def loss_func(v, rho, left_sites, proj_sites, right_sites):
    embed_v = embed_fp_even_vector(v)
    
    rho_lr = (
        rho
        & embed_v.reindex({f'k{i}': f'b{i}' for i in proj_sites})
        & embed_v.conj()
    )
    rho_lr = rho_lr.contract()
    tr_rho_lr = (
        rho_lr
        .reindex({f'k{i}': f'b{i}' for i in left_sites + right_sites})
        .contract()
    )
    
    rho_l = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in right_sites}
    )
    rho_l = rho_l.contract()*jnp.power(tr_rho_lr, -0.5)
    
    rho_r = rho_lr.reindex(
        {f'k{i}': f'b{i}' for i in left_sites}
    )
    rho_r = rho_r.contract()*jnp.power(tr_rho_lr, -0.5)
    
    purity_lr = get_rho_purity(rho_lr, left_sites + right_sites)
    purity_l = get_rho_purity(rho_l, left_sites)
    purity_r = get_rho_purity(rho_r, right_sites)
    
    reindex_map = (
        {f'k{i}': f'b{i}' for i in left_sites+right_sites}
        | {f'b{i}': f'k{i}' for i in left_sites+right_sites}
    )
    
    cross_term = (
        rho_lr.reindex(reindex_map)
        & rho_l
        & rho_r
    )
    cross_term = cross_term.contract()

    out = jnp.real(
        (purity_lr + purity_l*purity_r - 2*cross_term)/
        purity_lr
    )

    return out

## Extract cut rho and EDM

In [48]:
def generate_edm_from_cut_state(cut_state, sites, num_defect_sites):
    # Lots of duplicate code, probably a better way to do this.
    assert 2*num_defect_sites < len(sites)
    assert (num_defect_sites%2) == 0
    assert (len(sites)%2) == 0
    assert (sites[0]%2) == 0

    left_defect_sites = sites[:num_defect_sites]
    right_defect_sites = sites[-num_defect_sites:]
    internal_sites = sites[num_defect_sites:-num_defect_sites]

    # Being sloppy with the gate indices as the symmetries are invariant
    # under transpose and conjugation.
    left_sym_gates = [
        qu_M.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in left_defect_sites[::2]
    ]

    inner_gates = [
        qu_M.reindex({
            'ks': f'k{i}',
            'kf': f'k{i+1}',
            'bs': f'b{i}',
            'bf': f'b{i+1}'
        })
        for i in internal_sites[::2]
    ]

    right_sym_gates = [
        qu_M.reindex({
            'ks': f'b{i}',
            'kf': f'b{i+1}',
            'bs': f'c{i}',
            'bf': f'c{i+1}'
        })
        for i in right_defect_sites[::2]
    ]

    reindex_map = (
        {
            f'k{i}': f'c{i}'
            for i in (left_defect_sites + right_defect_sites)
        }
        |
        {
            f'k{i}': f'b{i}'
            for i in internal_sites
        }
    )

    # The fact that we don't conjugate the reindexed cut_state means that
    # we are effectively implementing local complex conjugation.
    edm = (
        cut_state
        & cut_state.reindex(reindex_map)
        & left_sym_gates
        & inner_gates
        & right_sym_gates
    )

    edm = edm.contract()

    fuse_maps = [
        ('k_left', (f'k{i}' for i in left_defect_sites)),
        ('b_left', (f'b{i}' for i in left_defect_sites)),
        ('k_right', (f'k{i}' for i in right_defect_sites)),
        ('b_right', (f'b{i}' for i in right_defect_sites))
    ]

    edm.fuse(fuse_maps, inplace=True)

    return edm

## Defect operators

In [49]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [50]:
def solve_for_boundary_operators(edm, num_iters=100):
    # Careful, the indices are reversed here for ease.
    # i.e. the k, b indices have been swapped to make tensor contraction easier.
    scores = list()

    u_left = qtn.tensor_builder.rand_tensor(
        (edm.ind_size('b_left'), edm.ind_size('k_left')),
        inds=['k_left', 'b_left'],
        dtype='complex64'
    )

    u_right = qtn.tensor_builder.rand_tensor(
        (edm.ind_size('b_right'), edm.ind_size('k_right')),
        inds=['k_right', 'b_right'],
        dtype='complex64'
    )

    for _ in range(num_iters):
        right_edm = (edm & u_left).contract()
        data = right_edm.data
        U, S, VH = np.linalg.svd(data)
        scores.append(np.sum(S))
    
        sol = (U @ VH).conj().T
        u_right = qtn.Tensor(sol, inds = ['b_right', 'k_right'])

        left_edm = (edm & u_right).contract()
        data = left_edm.data
        U, S, VH = np.linalg.svd(data)
        scores.append(np.sum(S))
    
        sol = (U @ VH).conj().T
        u_left = qtn.Tensor(sol, inds = ['b_left', 'k_left'])

    return (u_left, u_right), scores

## Apply random unitary to groundstate

In [51]:
def generate_random_su2():
    # Randomly sample a unitary, and scale by the determinant.
    u = unitary_group.rvs(2)
    det_u = np.linalg.det(u)
    su = u*np.power(det_u, -0.5)

    return su

In [52]:
X =  generate_random_su2()

In [53]:
np.linalg.det(X)

np.complex128(1+1.1102230246251568e-16j)

In [54]:
np.round(X @ (X.conj().T), 3)

array([[1.+0.j, 0.-0.j],
       [0.+0.j, 1.+0.j]])

In [55]:
def generate_random_symmetry_respecting_unitary_no_offset():
    # Generate a unitary which commutes with MK, where
    # M = X tensor (|0><0|) + Y tensor (|1><1|)

    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u0 = np.diag([phi_phasor, phi_phasor.conj()])

    if random() > 0.5:
        u0 = u0 @ np_X

    u1 = generate_random_su2()

    u = (
        tensor_product_operators(u0, np_00)
        + tensor_product_operators(u1, np_11)
    )

    qu_u = qtn.Tensor(
        u,
        inds=['ks', 'bs', 'kf', 'bf']
    )
    
    return qu_u

In [56]:
def generate_random_symmetry_respecting_unitary_offset():
    # Generate a unitary U such that IUI commutes with (MM)K, where
    # M = X tensor (|0><0|) + Y tensor (|1><1|), and concatenation denotes
    # tensor product.
    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u0 = np.diag([phi_phasor, phi_phasor.conj()])

    phi = np.random.uniform(0, 2*np.pi)
    phi_phasor = np.exp(1j*phi)
    u1 = np.diag([phi_phasor, phi_phasor.conj()])

    u = (
        tensor_product_operators(np_00, u0)
        + tensor_product_operators(np_11, u1)
    )

    qu_u = qtn.Tensor(
        u,
        inds=['kf', 'bf', 'ks', 'bs']
    )
    
    return qu_u

In [57]:
def generate_random_symmetry_respecting_unitary(offset):
    if offset:
        return generate_random_symmetry_respecting_unitary_offset()
    else:
        return generate_random_symmetry_respecting_unitary_no_offset()

In [58]:
# Warning, likely making assupmtions about shape of psi, number of sites being even here etc.
def apply_haar_random_fdlu_to_quimb_state(psi, domains_dict):
    num_sites = domains_dict['num_system_sites']

    depth = domains_dict['fdlu_depth']
    offset = domains_dict['fdlu_offset']
    all_circuit_lists = [
        list() for _ in range(depth)
    ]

    for layer, circuit_list in enumerate(all_circuit_lists):
        delta = layer
        is_offset = ((offset + delta)%2 == 1)

        for i in range(num_sites//2):
            site_1 = ((2*i)+delta+offset)%num_sites
            site_2 = ((2*i)+1+delta+offset)%num_sites

            u = generate_random_symmetry_respecting_unitary(is_offset)

            if is_offset:
                reindex_map = {
                    'kf': f'k_{layer+1}_{site_1}',
                    'ks': f'k_{layer+1}_{site_2}',
                    'bf': f'k_{layer}_{site_1}',
                    'bs': f'k_{layer}_{site_2}'
                }
            else:
                reindex_map = {
                    'ks': f'k_{layer+1}_{site_1}',
                    'kf': f'k_{layer+1}_{site_2}',
                    'bs': f'k_{layer}_{site_1}',
                    'bf': f'k_{layer}_{site_2}'
                }
            
            qu_u = u.reindex(reindex_map)

            circuit_list.append(qu_u)

    all_tensors = (
        [psi.reindex({f'k{i}': f'k_0_{i}' for i in range(num_sites)})]
        + sum(all_circuit_lists, start=[])
    )

    out = (
        qtn
        .TensorNetwork(all_tensors, virtual=False)
        .mangle_inner_()
        .reindex({f'k_{depth}_{i}': f'k{i}' for i in range(num_sites)}) 
    )

    return out

In [59]:
def extract_time_reversal_information_after_random_fdlu(psi, domains_dict,
    num_random_states=20):

    out = list()

    for _ in range(num_random_states):
        rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
        out.append(extract_time_reversal_information(rand_psi, domains_dict))

    return out

In [60]:
def extract_factorization_time_reversal_information_after_random_fdlu(psi,
    domains_dict, num_random_states=20):
    out = list()

    for _ in range(num_random_states):
        rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
        data = extract_factorization_time_reversal_information(
            rand_psi,
            domains_dict
        )
        out.append(data)

    return out

In [61]:
def get_quimb_psi_from_quspin_psi(quspin_psi):
    quimb_psi = qtn.Tensor(
        quspin_psi[::-1].reshape((2,)*num_sites),
        inds=[f'k{i}' for i in range(num_sites)]
    )

    return quimb_psi

## Sweep function

In [62]:
def extract_projector(psi, rho_sites, num_pad_sites, jw_even=False):
    proj_sites = rho_sites
    left_sites = list(range(
        min(rho_sites) - num_pad_sites,
        min(rho_sites)
    ))
    right_sites = list(range(
        max(rho_sites)+1,
        max(rho_sites)+num_pad_sites+1
    ))
    all_sites = left_sites + proj_sites + right_sites
    rho = (psi & psi.conj().reindex({f'k{i}': f'b{i}' for i in all_sites}))
    raw_proj_vec, proj_vec, losses = maximize_projector_states(
        rho,
        left_sites,
        proj_sites,
        right_sites
    )

    schmidt_vals = projector_state_check(
        proj_vec,
        rho,
        left_sites
    )

    return raw_proj_vec, proj_vec, losses, schmidt_vals

In [63]:
def find_invariants_via_projectors_from_random_state(psi, domains_dict, jw_even=False):
    rand_psi = apply_haar_random_fdlu_to_quimb_state(psi, domains_dict)
    
    raw_left_proj_vec, left_proj_vec, *left_proj_vec_results = extract_projector(
        rand_psi,
        domains_dict['left_projector_sites'],
        domains_dict['num_projector_pad_sites'],
        jw_even
    )
    
    raw_right_proj_vec, right_proj_vec, *right_proj_vec_results = extract_projector(
        rand_psi,
        domains_dict['right_projector_sites'],
        domains_dict['num_projector_pad_sites'],
        jw_even
    )
    
    cut_sites = list(range(
        min(domains_dict['left_projector_sites']),
        max(domains_dict['right_projector_sites'])+1
    ))
    
    cut_rho_unprojected = (
        rand_psi
        & rand_psi.conj().reindex({f'k{i}': f'b{i}' for i in cut_sites})
    )
    
    left_proj_sites = domains_dict['left_projector_sites']
    right_proj_sites = domains_dict['right_projector_sites']
    
    cut_rho = (
        cut_rho_unprojected.reindex({
            **{f'k{i}': f'l{i}' for i in left_proj_sites},
            **{f'k{i}': f'l{i}' for i in right_proj_sites},
            **{f'b{i}': f'c{i}' for i in left_proj_sites},
            **{f'b{i}': f'c{i}' for i in right_proj_sites}
        })
        & left_proj_vec.conj().reindex({f'k{i}': f'l{i}' for i in left_proj_sites})
        & left_proj_vec
        & left_proj_vec.reindex({f'k{i}': f'c{i}' for i in left_proj_sites})
        & left_proj_vec.conj().reindex({f'k{i}': f'b{i}' for i in left_proj_sites})
        & right_proj_vec.conj().reindex({f'k{i}': f'l{i}' for i in right_proj_sites})
        & right_proj_vec
        & right_proj_vec.reindex({f'k{i}': f'c{i}' for i in right_proj_sites})
        & right_proj_vec.conj().reindex({f'k{i}': f'b{i}' for i in right_proj_sites})
    )
    
    cut_rho_trace = (
        cut_rho
        .reindex({f'b{i}': f'k{i}' for i in cut_sites})
        .contract()
    )
    
    cut_rho = cut_rho/cut_rho_trace
    
    tranpose_map = (
        {f'k{i}': f'b{i}' for i in cut_sites}
        | {f'b{i}': f'k{i}' for i in cut_sites}
    )
    
    cut_rho_purity = (
        (cut_rho & cut_rho.reindex(tranpose_map))
        .contract()
    )
    
    sub_cut_sites = list(range(
        max(domains_dict['left_projector_sites'])+1,
        min(domains_dict['right_projector_sites'])
    ))
    
    sub_cut_rho = (
        cut_rho
        .reindex({f'k{i}': f'b{i}' for i in left_proj_sites + right_proj_sites})
        .contract()
    )
    
    sub_cut_rho_trace = (
        sub_cut_rho
        .reindex({f'b{i}': f'k{i}' for i in sub_cut_sites})
        .contract()
    )
    
    tranpose_map = (
        {f'k{i}': f'b{i}' for i in sub_cut_sites}
        | {f'b{i}': f'k{i}' for i in sub_cut_sites}
    )
    
    sub_cut_rho_purity = (
        (sub_cut_rho & sub_cut_rho.reindex(tranpose_map))
        .contract()
    )
    
    cut_score, cut_state, cut_overlap = get_dominant_eigenvector(sub_cut_rho, False)

    cut_state_fermion_parity = compute_fermion_parity(cut_state, sub_cut_sites)

    edm = generate_edm_from_cut_state(
        cut_state,
        sub_cut_sites,
        domains_dict['num_defect_sites']
    )

    defect_ops_results = solve_for_boundary_operators(
        edm,
        num_iters=20
    )

    left_defect_sites = list(range(
        max(domains_dict['left_projector_sites']) + 1,
        max(domains_dict['left_projector_sites']) + 1 + domains_dict['num_defect_sites']
    ))
    right_defect_sites = list(range(
        min(domains_dict['right_projector_sites']) - domains_dict['num_defect_sites'],
        min(domains_dict['right_projector_sites'])
    ))
    
    left_rdm = (
        cut_rho
        .reindex({
            f'b{i}': f'k{i}'
            for i in cut_sites if i not in left_defect_sites
        })
    )
    left_rdm = left_rdm.contract()
    left_fuse_map = [
        ('k_left', [f'k{i}' for i in left_defect_sites]),
        ('b_left', [f'b{i}' for i in left_defect_sites])
    ]
    left_rdm.fuse(left_fuse_map, inplace=True)
    
    right_rdm = (
        cut_rho
        .reindex({
            f'b{i}': f'k{i}'
            for i in cut_sites if i not in right_defect_sites
        })
    )
    right_rdm = right_rdm.contract()
    right_fuse_map = [
        ('k_right', [f'k{i}' for i in right_defect_sites]),
        ('b_right', [f'b{i}' for i in right_defect_sites])
    ]
    right_rdm.fuse(right_fuse_map, inplace=True)

    left_defect_op, right_defect_op = defect_ops_results[0]

    np_left_rdm = (
        left_rdm
        .transpose('k_left', 'b_left')
        .data
    )

    np_left_defect_op = (
        left_defect_op
        .transpose('k_left', 'b_left')
        .data
        .T
    )

    fp_list = [np_I, np_Z]
    np_fp = multikron([
        fp_list[i%2]
        for i in range(domains_dict['num_defect_sites'])
    ])

    left_defect_op_invariant = np.trace(
        np_fp
        @ np_left_defect_op.conj().T
        @ np_fp
        @ np_left_defect_op
        @ np_left_rdm
    )

    np_right_rdm = (
        right_rdm
        .transpose('k_right', 'b_right')
        .data
    )
    
    np_right_defect_op = (
        right_defect_op
        .transpose('k_right', 'b_right')
        .data
        .T
    )

    right_defect_op_invariant = np.trace(
        np_fp
        @ np_right_defect_op.conj().T
        @ np_fp
        @ np_right_defect_op
        @ np_right_rdm
    )

    np_sym_defect_m = get_symmetry_defect_m(domains_dict['num_defect_sites'])

    left_cocyles = get_cocycles_from_rho(
        np_left_rdm,
        np_left_defect_op,
        np_sym_defect_m
    )

    right_cocyles = get_cocycles_from_rho(
        np_right_rdm,
        np_right_defect_op,
        np_sym_defect_m
    )

    all_cocyles = np.array([left_cocyles, right_cocyles])

    cocycle_equation_output = compute_cocyle_equation_from_cocycles(all_cocyles)

    cocycle_invariants = (all_cocyles[:,0,0]**2)*(all_cocyles[:,1,1])

    out = {
        'left_proj_vec': left_proj_vec,
        'raw_left_proj_vec': raw_left_proj_vec,
        'left_proj_vec_results': left_proj_vec_results,
        'right_proj_vec': right_proj_vec,
        'raw_right_proj_vec': raw_right_proj_vec,
        'right_proj_vec_results': right_proj_vec_results,
        'cut_rho_trace': cut_rho_trace,
        'cut_rho_purity': cut_rho_purity,
        'sub_cut_rho_trace': sub_cut_rho_trace,
        'sub_cut_rho_purity': sub_cut_rho_purity,
        'cut_score': cut_score,
        'cut_state': cut_state,
        'cut_state_fermion_parity': cut_state_fermion_parity,
        'cut_overlap': cut_overlap,
        'defect_ops_scores': defect_ops_results[1],
        'left_defect_op': left_defect_op,
        'right_defect_op': right_defect_op,
        'left_defect_op_invariant': left_defect_op_invariant,
        'right_defect_op_invariant': right_defect_op_invariant,
        'cocycles': all_cocyles,
        'cocycle_equation_output': cocycle_equation_output,
        'cocycle_invariants': cocycle_invariants
    }

    return out

## Find dominant eigenvector

In [64]:
def random_uniform_complex(shape):
    return np.random.uniform(size=shape) + 1j*np.random.uniform(size=shape)

In [65]:
def lanczos_iteration(rho, v, sites):
    w = (
        rho & v.reindex({f'k{i}': f'b{i}' for i in sites})
    ).contract()

    w_norm = np.sqrt((w & w.conj()).contract())

    out = w/w_norm

    return out

In [66]:
def multiply_state_by_jw_string(state, sites):
    gates = [
        qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
        for i in sites if (i%2 == 1)
    ]

    out = qtn.TensorNetwork(
        [
            *gates,
            state.reindex({f'k{i}': f'b{i}' for i in sites if (i%2 == 1)})
        ]
    )

    return out.contract()

In [67]:
def lanczos_iteration_jw_even(rho, v, sites):
    w = multiply_state_by_jw_string(v, sites)
    w = (v+w)/2

    w = (
        rho & w.reindex({f'k{i}': f'b{i}' for i in sites})
    ).contract()

    w = (multiply_state_by_jw_string(w, sites) + w)/2

    w_norm = np.sqrt((w & w.conj()).contract())

    out = w/w_norm

    return out

In [68]:
def lanczos_algorithm(rho, v, sites, num_iters=20, jw_even=False):
    update_func = lanczos_iteration_jw_even if jw_even else lanczos_iteration
    for _ in range(num_iters):
        v = update_func(rho, v, sites)

    return v

In [69]:
def get_dominant_eigenvector(rho, jw_even=False):
    k_inds = [i for i in rho.inds if i.startswith('k')]
    #b_inds = [i for i in rho.inds if i.startswith('b')]

    sites = [int(s[1:]) for s in k_inds]

    v = qtn.Tensor(
        data=random_uniform_complex((2,)*len(sites)),
        inds=[f'k{i}' for i in sites]
    )

    v = lanczos_algorithm(
        rho,
        v,
        sites,
        num_iters=20,
        jw_even=jw_even
    )

    update_func = lanczos_iteration_jw_even if jw_even else lanczos_iteration
    new_v = update_func(rho, v, sites)

    overlap = np.abs((v & new_v.conj()).contract())

    score = (
        v.reindex({f'k{i}': f'b{i}' for i in sites})
        & rho
        & v.conj()
    )
    score = score.contract()


    return score, v, overlap

In [70]:
def compute_fermion_parity(state, sites):
    odd_sites = [i for i in sites if (i%2 == 1)]

    gates = [
        qu_Z.reindex({'k': f'k{i}', 'b': f'b{i}'})
        for i in odd_sites
    ]

    gate_exp = (
        state.reindex({f'k{i}': f'b{i}' for i in odd_sites})
        & gates
        & state.conj()
    ).contract()

    return gate_exp

## Extract cocyles

In [71]:
def get_symmetry_defect_m(num_defect_sites):
    assert (num_defect_sites%2 == 0)

    np_op = multikron([np_M_reindexed]*(num_defect_sites//2))

    out = qtn.Tensor(
        np_op,
        inds=['k', 'b'],
        tags='sym_defect_M',
    )

    out = out.transpose('k', 'b').data

    return out

In [72]:
def get_t_squared_defect_op_from_t_defect_op(defect_op, unitary_sym_op):
    out = (
        unitary_sym_op
        @ (defect_op.conj())
        @ (unitary_sym_op.conj().T)
        @ defect_op
    )

    return out

In [73]:
def get_t_inverse_defect_op_from_t_defect_op(defect_op, unitary_sym_op):
    out = (
        (unitary_sym_op.T)
        @ (defect_op.T)
        @ (unitary_sym_op.conj())
    )

    return out

In [74]:
def get_all_defect_ops_from_t_defect_op(defect_op, unitary_sym_op):
    out = [
        np.identity(defect_op.shape[0]),
        defect_op,
        get_t_squared_defect_op_from_t_defect_op(defect_op, unitary_sym_op),
        get_t_inverse_defect_op_from_t_defect_op(defect_op, unitary_sym_op)
    ]

    return out

In [75]:
def get_cocycle(defect_rho, defect_op_list, unitary_sym_op, g_index,
                  h_index):
    g_op = defect_op_list[g_index]

    h_op = defect_op_list[h_index]
    h_op = h_op.conj() if (g_index%2 == 1) else h_op

    gh_index = (g_index+h_index)%4
    gh_op = defect_op_list[gh_index]

    out = np.trace(
        (gh_op.conj().T)
        @ unitary_sym_op
        @ h_op
        @ (unitary_sym_op.conj().T)
        @ g_op
        @ defect_rho
    )

    return out

In [76]:
def get_cocycles_from_rho(defect_rho, defect_op, unitary_sym_op):
    defect_op_list = get_all_defect_ops_from_t_defect_op(
        defect_op,
        unitary_sym_op
    )

    out = [
        [
            get_cocycle(defect_rho, defect_op_list, unitary_sym_op, i, j)
            for i in range(1,4)
        ]
        for j in range(1,4)
    ]

    return out

In [77]:
def compute_cocyle_equation_from_cocycles(cocyles):
    shape = cocyles
    # A bit janky, oh well
    X = np.ones((2, 4, 4), dtype=complex)

    X[:, 1:, 1:] = cocyles

    out = np.zeros((2, 4, 4, 4), dtype=complex)

    for i,j,k in product(range(4), repeat=3):
        num = X[:,i,j]*X[:,(i+j)%4, k]
        if (i%2 == 0):
            denom = X[:,j,k]*X[:, i, (j+k)%4]
        if (i%2 == 1):
            denom = (X[:,j,k].conj())*X[:,i,(j+k)%4]

        #print(num)
        #print(denom)

        out[:,i,j,k] = num/denom

    return out

# Check random unitaries and symmetry

In [78]:
domains_dict = {
    'num_system_sites': 24,
    'left_projector_sites': list(range(4, 8)),
    'right_projector_sites': list(range(16, 20)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 2,
    'fdlu_offset': 1
}

In [79]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [80]:
rand_psi = apply_haar_random_fdlu_to_quimb_state(cluster_psi, domains_dict)

In [81]:
(rand_psi & rand_psi.conj()).contract()

np.complex128(0.9999999999999973-4.5102810375396984e-17j)

In [82]:
(rand_psi & cluster_psi.conj()).contract()

np.complex128(-0.0001212520194105122+1.1519648082658485e-19j)

In [83]:
symmetry_gates = [
    qu_M.reindex({
        'ks': f'k{i}', 'bs':f'b{i}',
        'kf': f'k{i+1}', 'bf':f'b{i+1}',
    })
    for i in range(0, domains_dict['num_system_sites'], 2)
]

In [84]:
sym_cluster_psi = qtn.TensorNetwork(
    [
        rand_psi.reindex({f'k{i}': f'b{i}' for i in range(domains_dict['num_system_sites'])}),
        *symmetry_gates
    ]
)

In [85]:
(
    sym_cluster_psi & sym_cluster_psi.conj()
).contract()

np.complex128(0.9999999999999971-3.8163916471489756e-17j)

In [86]:
(
    sym_cluster_psi & rand_psi.conj()
).contract()

np.complex128(0.00017771200543805664+9.825582188149884e-20j)

In [87]:
(
    sym_cluster_psi & rand_psi
).contract()

np.complex128(0.9999999999999974+5.551115123125783e-17j)

In [88]:
product_psi = get_product_qu_tensor_network(domains_dict['num_system_sites'])

In [89]:
rand_psi = apply_haar_random_fdlu_to_quimb_state(product_psi, domains_dict)

In [90]:
(rand_psi & rand_psi.conj()).contract()

np.complex128(0.9999999999999969+0j)

In [91]:
(rand_psi & product_psi.conj()).contract()

np.complex128(-0.0049959137616231875+1.4775976926352318e-18j)

In [92]:
symmetry_gates = [
    qu_M.reindex({
        'ks': f'k{i}', 'bs':f'b{i}',
        'kf': f'k{i+1}', 'bf':f'b{i+1}',
    })
    for i in range(0, domains_dict['num_system_sites'], 2)
]

In [93]:
sym_product_psi = qtn.TensorNetwork(
    [
        rand_psi.reindex({f'k{i}': f'b{i}' for i in range(domains_dict['num_system_sites'])}),
        *symmetry_gates
    ]
)

In [94]:
(
    sym_product_psi & sym_product_psi.conj()
).contract()

np.complex128(0.9999999999999969+0j)

In [95]:
(
    sym_product_psi & rand_psi.conj()
).contract()

np.complex128(-0.0022668081726628323-4.7704895589362195e-18j)

In [96]:
(
    sym_product_psi & rand_psi
).contract()

np.complex128(0.9999999999999969+0j)

So random unitaries are symmetric, nice.

# Test - Cluster state

In [97]:
domains_dict = {
    'num_system_sites': 24,
    'left_projector_sites': list(range(4, 8)),
    'right_projector_sites': list(range(16, 20)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 0,
    'fdlu_offset': 0
}

In [98]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [99]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:13<00:00,  1.51it/s]


### Analyze results

#### Projector scores

In [100]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,
        0.,  0.,  0., -0.,  0., -0.,  0.])

In [101]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0.,  0.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,  0., -0., -0.,
        0., -0.,  0.,  0., -0.,  0.,  0.])

In [102]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [103]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

In [104]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

#### Purities

In [105]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j])

In [106]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j])

In [107]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j])

Purities are the same, which one could likely prove.

In [108]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j])

#### Cut state results

In [109]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([ 1.-3.78640785e-18j, -1.+3.81772135e-18j, -1.+1.42758190e-17j,
        1.+1.03101589e-17j, -1.-1.50316029e-17j, -1.-2.79993761e-18j,
       -1.-1.66589314e-17j,  1.-2.06270653e-17j, -1.+1.51270839e-18j,
       -1.+1.93261934e-17j,  1.+8.55485492e-18j,  1.+1.28677590e-17j,
        1.-1.62473690e-17j, -1.-3.66893593e-18j, -1.-6.25862832e-18j,
        1.-7.38922396e-18j, -1.-3.18675889e-18j,  1.+9.93683909e-18j,
       -1.-4.98380590e-18j,  1.+5.22413751e-18j])

So the FP of the cut state can be even or odd. Interesting!

In [110]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [111]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [112]:
np.array([d['defect_ops_scores'][-1] for d in results])

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [113]:
[d['defect_ops_scores'][-1] for d in results]

[np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(0.9999999999999998),
 np.float64(0.9999999999999998),
 np.float64(0.9999999999999998),
 np.float64(0.9999999999999996),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(0.9999999999999998)]

#### Phases

In [114]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [115]:
left_phases.shape

(20,)

In [116]:
np.round(left_phases, 3)

array([-1.-0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.-0.j,
       -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j,
       -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j])

In [117]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [118]:
right_phases.shape

(20,)

In [119]:
np.round(right_phases, 3)

array([-1.-0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.-0.j,
       -1.-0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.-0.j,
       -1.-0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.-0.j, -1.-0.j])

#### Cocyle information

In [120]:
np.array([
    d['cocycle_invariants'] for d in results
])

array([[ 1.38777878e-17-1.92593006e-34j,  6.93480707e-01+1.00321327e-16j],
       [ 6.91473864e-01+1.70664680e-16j, -9.78741860e-01+1.89162126e-16j],
       [-2.65149595e-02+2.81971384e-16j,  1.17399226e-01+8.87402988e-18j],
       [ 1.11053642e-01+3.63919196e-16j,  1.52660830e-16-3.09942969e-32j],
       [-1.11022302e-16-1.11022303e-16j,  3.91865123e-16+5.55111512e-17j],
       [ 9.90000107e-01-5.55112522e-17j,  4.96412589e-16-5.55111512e-17j],
       [-1.21430643e-16-1.34815155e-32j,  2.08892804e-16+4.72924622e-32j],
       [ 9.52583286e-01-5.95341586e-26j, -1.06428489e-01-2.07201915e-17j],
       [-2.01088089e-02-3.30834381e-16j,  9.94100403e-01-7.16324642e-16j],
       [-5.55111512e-17+5.55111512e-17j, -5.51120077e-17+5.55111512e-17j],
       [-1.66533454e-16-4.44089210e-16j,  1.39293402e-01-2.22777225e-17j],
       [ 1.28455527e-01+1.42614357e-17j, -1.79735830e-16+1.11022302e-16j],
       [-1.11022302e-16+3.33066907e-16j,  1.08723483e-16-3.88578059e-16j],
       [ 4.47213595e-01-2

In [133]:
X = results[0]['cocycle_equation_output']

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [134]:
X.shape

(2, 4, 4, 4)

In [135]:
np.round(X, 3)

array([[[[ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j],
         [ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00-0.00000000e+00j],
         [ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00-0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j],
         [ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00-0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j]],

        [[ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00-0.00000000e+00j],
         [ 1.00000000e+00+0.00000000e+00j,
           3.34760030e+15+1.80568156e+16j,
           4.60000000e-02-2.51000000e-01j,
    

In [139]:
X[:, 0]

array([[[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]],

       [[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]]])

In [140]:
X[:, : ,0]

array([[[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]],

       [[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]]])

In [141]:
X[:, : , :, 0]

array([[[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]],

       [[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]]])

In [142]:
sub_X = X[:, 1:, 1:, 1:]

In [146]:
sub_X.shape

(2, 3, 3, 3)

In [145]:
sub_X[0]

array([[[ 3.34760030e+15+1.80568156e+16j,
          4.64572866e-02-2.50588655e-01j,
         -1.00000000e+00+1.38777882e-17j],
        [-9.92602073e-18+5.35405396e-17j,
          2.80627948e-16-2.02460331e-16j,
         -9.92602073e-18+5.35405395e-17j],
        [-1.00000000e+00+4.27546122e-33j,
          8.19936485e+15+7.83227884e+15j,
          3.34760030e+15+1.80568156e+16j]],

       [[-7.20575940e+16-2.92601747e-08j,
          8.86475203e-01-1.35543360e+00j,
          1.00000000e+00-1.38777882e-17j],
        [-1.38777878e-17-1.92592994e-34j,
          1.00000000e+00+0.00000000e+00j,
         -1.38777878e-17-1.92592994e-34j],
        [ 1.00000000e+00-1.38777882e-17j,
          3.37957370e-01+5.16741780e-01j,
         -7.20575940e+16-2.93948689e-08j]],

       [[-8.19936485e+15+7.83227884e+15j,
         -3.34760030e+15+1.80568156e+16j,
         -1.00000000e+00+4.27546122e-33j],
        [ 6.37714751e-17+6.09164227e-17j,
          2.80627948e-16-2.02460331e-16j,
          6.37714751e-1

In [124]:
X = results[0]['cocycles']

In [125]:
X.shape

(2, 3, 3)

In [126]:
np.round(X[0], 3)

array([[ 1.-0.j, -0.+0.j, -1.-0.j],
       [-1.+0.j,  0.+0.j,  1.-0.j],
       [ 1.-0.j, -0.-0.j,  1.-0.j]])

In [127]:
np.round(X[1], 3)

array([[ 1.   +0.j  , -0.417-0.82j, -1.   -0.j  ],
       [-1.   -0.j  ,  0.693+0.j  ,  1.   +0.j  ],
       [ 1.   +0.j  , -0.417-0.82j,  1.   +0.j  ]])

In [128]:
X = results[1]['cocycles']

In [129]:
X.shape

(2, 3, 3)

In [130]:
np.round(X[0], 3)

array([[ 1.   +0.j   , -0.546-0.715j, -1.   +0.j   ],
       [-1.   -0.j   ,  0.691+0.j   ,  1.   +0.j   ],
       [ 1.   -0.j   , -0.546-0.715j,  1.   +0.j   ]])

In [131]:
np.round(X[1], 3)

array([[ 1.   -0.j   ,  0.087-0.055j, -1.   +0.j   ],
       [-1.   +0.j   , -0.979+0.j   ,  1.   -0.j   ],
       [ 1.   -0.j   ,  0.087-0.055j,  1.   -0.j   ]])

# Test - Cluster state - 2 site projectors and defect operators

In [134]:
domains_dict = {
    'num_system_sites': 24,
    'left_projector_sites': list(range(6, 8)),
    'right_projector_sites': list(range(16, 18)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 0,
    'fdlu_offset': 0
}

In [135]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [137]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:12<00:00,  1.56it/s]


### Analyze results

#### Projector scores

In [138]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

In [139]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0.,  0.,  0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
        0.,  0.,  0.,  0.,  0.,  0.,  0.])

In [140]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [141]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

In [142]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

#### Purities

In [143]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])

In [144]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j])

In [145]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j])

Purities are the same, which one could likely prove.

In [146]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j])

#### Cut state results

In [147]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([ 1.+1.26641750e-17j,  1.-1.88734595e-17j, -1.+1.64012623e-17j,
       -1.-9.77861690e-18j,  1.-8.31959065e-18j, -1.-1.63685256e-17j,
        1.-1.17775276e-17j,  1.+3.68607604e-18j, -1.+1.61124511e-17j,
       -1.-7.19220486e-18j, -1.-1.26801094e-17j,  1.-2.19899192e-18j,
        1.-3.52572227e-18j,  1.+5.09434847e-18j, -1.+1.98617395e-17j,
       -1.+1.17428872e-17j,  1.-1.65180155e-18j,  1.-1.26482542e-17j,
       -1.+5.28304173e-18j, -1.+1.38585998e-17j])

So the FP of the cut state can be even or odd. Interesting!

In [148]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [149]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [150]:
np.array([d['defect_ops_scores'][-1] for d in results])

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [151]:
[d['defect_ops_scores'][-1] for d in results]

[np.float64(1.0000000000000004),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(0.9999999999999998),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(0.9999999999999998),
 np.float64(0.9999999999999998),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(0.9999999999999998),
 np.float64(0.9999999999999998),
 np.float64(1.0000000000000004),
 np.float64(1.0000000000000002)]

#### Phases

In [152]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [153]:
left_phases.shape

(20,)

In [154]:
np.round(left_phases, 3)

array([-1.-0.j, -1.-0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.-0.j, -1.+0.j,
       -1.+0.j, -1.+0.j, -1.-0.j, -1.-0.j, -1.-0.j, -1.-0.j, -1.-0.j,
       -1.-0.j, -1.-0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.-0.j])

In [155]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [156]:
right_phases.shape

(20,)

In [157]:
np.round(right_phases, 3)

array([-1.-0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.-0.j,
       -1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j,
       -1.-0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j])

#### Cocyle information

In [158]:
np.array([
    d['cocycle_invariants'] for d in results
])

array([[ 1.94289029e-16+4.31408308e-32j, -4.61661152e-01-2.64786355e-16j],
       [ 2.02750350e-03+5.57362494e-17j, -1.66533454e-16-2.22044605e-16j],
       [ 1.88283746e-01-2.01140910e-16j, -9.29515876e-01+1.58708144e-16j],
       [-5.55111512e-17-2.77555756e-16j,  5.55111512e-17-4.44089210e-16j],
       [-7.53789577e-02+2.34597723e-16j, -1.42247325e-16+1.11022302e-16j],
       [ 9.39765971e-03+1.11022302e-16j, -8.14742487e-01-2.67514741e-26j],
       [-1.38777878e-16+2.22044605e-16j, -7.14616586e-01+3.96691894e-17j],
       [ 3.97804639e-01-9.96763382e-17j, -5.55111512e-17+0.00000000e+00j],
       [-9.02056208e-17-1.49235048e-42j, -1.17961196e-16-1.39148438e-32j],
       [-1.66533454e-16-2.22044605e-16j, -4.06511236e-33+2.77555756e-16j],
       [-8.32667268e-17+1.68155398e-44j, -1.24900090e-16-1.11022302e-16j],
       [ 1.11022302e-16+4.44089216e-16j, -2.60382304e-01+4.33623644e-17j],
       [ 8.91048122e-04+3.88528609e-16j,  1.11022302e-16-4.99600361e-16j],
       [-3.46944695e-17+3

In [159]:
X = results[0]['cocycle_equation_output']

In [160]:
X.shape

(2, 4, 4, 4)

In [161]:
np.round(X, 3)

array([[[[ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j],
         [ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00-0.00000000e+00j,
           1.00000000e+00-0.00000000e+00j],
         [ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00-0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j],
         [ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00-0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j]],

        [[ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00-0.00000000e+00j,
           1.00000000e+00-0.00000000e+00j],
         [ 1.00000000e+00+0.00000000e+00j,
           6.32980293e+14-4.32988567e+15j,
           1.23000000e-01+8.41000000e-01j,
    

In [162]:
X[:, 0]

array([[[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]],

       [[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]]])

In [163]:
X[:, : ,0]

array([[[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]],

       [[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]]])

In [164]:
X[:, : , :, 0]

array([[[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]],

       [[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]]])

In [165]:
sub_X = X[:, 1:, 1:, 1:]

In [166]:
sub_X.shape

(2, 3, 3, 3)

In [167]:
sub_X[0]

array([[[ 6.32980293e+14-4.32988567e+15j,
          1.22981127e-01+8.41249284e-01j,
         -1.00000000e+00-3.33066907e-16j],
        [-3.30562628e-17-2.26120529e-16j,
         -1.49233826e-16+1.37548279e-16j,
         -3.30562628e-17-2.26120529e-16j],
        [-1.00000000e+00-2.22044605e-16j,
          3.27002092e+15+4.78477348e+15j,
          6.32980293e+14-4.32988567e+15j]],

       [[-5.14697100e+15-5.71428571e-01j,
         -6.78456459e-01+3.31381732e-01j,
          1.00000000e+00+1.11022302e-16j],
        [-1.94289029e-16+4.31408308e-32j,
          1.00000000e+00+0.00000000e+00j,
         -1.94289029e-16+4.31408308e-32j],
        [ 1.00000000e+00+1.11022302e-16j,
         -1.19003018e+00-5.81252131e-01j,
         -5.14697100e+15-5.71428571e-01j]],

       [[-3.27002092e+15+4.78477348e+15j,
         -6.32980293e+14-4.32988567e+15j,
         -1.00000000e+00-2.22044605e-16j],
        [ 9.73594474e-17+1.42458692e-16j,
         -1.49233826e-16+1.37548279e-16j,
          9.73594474e-1

In [168]:
X = results[0]['cocycles']

In [169]:
X.shape

(2, 3, 3)

In [170]:
np.round(X[0], 3)

array([[ 1.+0.j, -0.-0.j, -1.-0.j],
       [-1.-0.j,  0.+0.j,  1.+0.j],
       [ 1.+0.j, -0.-0.j,  1.+0.j]])

In [171]:
np.round(X[1], 3)

array([[ 1.   +0.j   , -0.509+0.101j, -1.   -0.j   ],
       [-1.   -0.j   , -0.462-0.j   ,  1.   +0.j   ],
       [ 1.   +0.j   , -0.509+0.101j,  1.   +0.j   ]])

In [172]:
X = results[1]['cocycles']

In [173]:
X.shape

(2, 3, 3)

In [174]:
np.round(X[0], 3)

array([[ 1.   +0.j   , -0.078+0.033j, -1.   -0.j   ],
       [-1.   -0.j   ,  0.002+0.j   ,  1.   +0.j   ],
       [ 1.   +0.j   , -0.078+0.033j,  1.   +0.j   ]])

In [175]:
np.round(X[1], 3)

array([[ 1.-0.j,  0.+0.j, -1.+0.j],
       [-1.-0.j, -0.-0.j,  1.-0.j],
       [ 1.+0.j,  0.+0.j,  1.-0.j]])

#### Local projectors

In [178]:
np.round(results[0]['left_proj_vec'].data, 3)

array([[-0.   -0.j   ,  0.   +0.j   ],
       [ 0.746+0.666j,  0.   +0.j   ]])

In [180]:
np.round(results[0]['right_proj_vec'].data, 3)

array([[-0.   -0.j   ,  0.   +0.j   ],
       [ 0.487+0.873j,  0.   +0.j   ]])

In [181]:
np.round(results[1]['left_proj_vec'].data, 3)

array([[0.787+0.617j, 0.   +0.j   ],
       [0.   +0.j   , 0.   +0.j   ]])

In [182]:
np.round(results[1]['right_proj_vec'].data, 3)

array([[0.98+0.201j, 0.  +0.j   ],
       [0.  +0.j   , 0.  +0.j   ]])

Let's use this one, projectors of the form $|00><00|$, which we understand well.

##### Defect operators

In [195]:
left_defect_sites = [8,9]
right_defect_sites = [14,15]

In [217]:
left_fuse_map = [
    ('k_left', [f'k{i}' for i in left_defect_sites]),
    ('b_left', [f'b{i}' for i in left_defect_sites])
]

right_fuse_map = [
    ('k_right', [f'k{i}' for i in right_defect_sites]),
    ('b_right', [f'b{i}' for i in right_defect_sites])
]

left_shape_map = {
    'k_left': [2,2],
    'b_left': [2,2]
}

right_shape_map = {
    'k_right': [2,2],
    'b_right': [2,2]
}

In [224]:
results[1]['left_defect_op'].unfuse(left_fuse_map, shape_map=left_shape_map)

Tensor(shape=(2, 2, 2, 2), inds=('b8', 'b9', 'k8', 'k9'), tags=oset([]))

In [225]:
np.round(
    results[1]['left_defect_op']
    .unfuse(left_fuse_map, shape_map=left_shape_map)
    .transpose('k8', 'k9', 'b8', 'b9')
    .data,
3)

array([[[[ 0.   +0.j   , -0.   +0.j   ],
         [-0.   -0.j   ,  0.893-0.449j]],

        [[-0.   +0.j   ,  0.   +0.j   ],
         [ 0.893-0.449j,  0.   -0.j   ]]],


       [[[ 0.789-0.604j,  0.111+0.j   ],
         [ 0.   -0.j   ,  0.   +0.j   ]],

        [[ 0.018+0.11j ,  0.466-0.878j],
         [ 0.   +0.j   ,  0.   -0.j   ]]]])

In [226]:
np.round(
    results[1]['right_defect_op']
    .unfuse(right_fuse_map, shape_map=right_shape_map)
    .transpose('k14', 'k15', 'b14', 'b15')
    .data,
3)

array([[[[-0.   -0.j   ,  0.484-0.875j],
         [ 0.   +0.j   ,  0.   -0.j   ]],

        [[-0.   +0.j   ,  0.   -0.j   ],
         [-0.   -0.j   , -0.465+0.885j]]],


       [[[-0.203-0.979j, -0.   +0.j   ],
         [ 0.   -0.j   , -0.   +0.j   ]],

        [[ 0.   +0.j   , -0.   +0.j   ],
         [ 0.484-0.875j, -0.   +0.j   ]]]])

In [233]:
cut_psi = results[1]['cut_state']

In [234]:
cut_psi

Tensor(shape=(2, 2, 2, 2, 2, 2, 2, 2), inds=('k8', 'k10', 'k12', 'k14', 'k9', 'k11', 'k13', 'k15'), tags=oset(['prod', 'CZ', 'Had', 'Z0_pad']))

In [235]:
right_rdm = (
    cut_psi
    & cut_psi.conj().reindex({'k14': 'b14', 'k15': 'b15'})
)

right_rdm = right_rdm.contract()

In [236]:
np.round(right_rdm.transpose('k14', 'k15', 'b14', 'b15').data, 3)

array([[[[0.5+0.j, 0. +0.j],
         [0. +0.j, 0. +0.j]],

        [[0. -0.j, 0. +0.j],
         [0. +0.j, 0. -0.j]]],


       [[[0. -0.j, 0. +0.j],
         [0. +0.j, 0. -0.j]],

        [[0. +0.j, 0. +0.j],
         [0. +0.j, 0.5+0.j]]]])

# Debug - plug in expected defect operators

In [245]:
cut_state = results[1]['cut_state']

In [246]:
results[1]['left_defect_op']

Tensor(shape=(4, 4), inds=('b_left', 'k_left'), tags=oset([]))

In [247]:
results[1]['right_defect_op']

Tensor(shape=(4, 4), inds=('b_right', 'k_right'), tags=oset([]))

In [260]:
left_defect_op = qtn.Tensor(
    np.kron(np_X, np_X),
    inds=['k_left', 'b_left']
)

right_defect_op = qtn.Tensor(
    np.kron(np_I, np_X),
    inds=['k_right', 'b_right']
)

In [261]:
sub_cut_sites = list(range(
    max(domains_dict['left_projector_sites'])+1,
    min(domains_dict['right_projector_sites'])
))

In [262]:
rand_psi = cluster_psi

In [263]:
cut_state_fermion_parity = compute_fermion_parity(cut_state, sub_cut_sites)

edm = generate_edm_from_cut_state(
    cut_state,
    sub_cut_sites,
    domains_dict['num_defect_sites']
)

defect_ops_results = solve_for_boundary_operators(
    edm,
    num_iters=20
)

left_defect_sites = list(range(
    max(domains_dict['left_projector_sites']) + 1,
    max(domains_dict['left_projector_sites']) + 1 + domains_dict['num_defect_sites']
))
right_defect_sites = list(range(
    min(domains_dict['right_projector_sites']) - domains_dict['num_defect_sites'],
    min(domains_dict['right_projector_sites'])
))

left_rdm = (
    cut_state
    & cut_state.conj().reindex({
        f'k{i}': f'b{i}'
        for i in left_defect_sites
    })
)
left_rdm = left_rdm.contract()
left_fuse_map = [
    ('k_left', [f'k{i}' for i in left_defect_sites]),
    ('b_left', [f'b{i}' for i in left_defect_sites])
]
left_rdm.fuse(left_fuse_map, inplace=True)

right_rdm = (
    cut_state
    & cut_state.conj().reindex({
        f'k{i}': f'b{i}'
        for i in right_defect_sites
    })
)
right_rdm = right_rdm.contract()
right_fuse_map = [
    ('k_right', [f'k{i}' for i in right_defect_sites]),
    ('b_right', [f'b{i}' for i in right_defect_sites])
]
right_rdm.fuse(right_fuse_map, inplace=True)

#left_defect_op, right_defect_op = defect_ops_results[0]

np_left_rdm = (
    left_rdm
    .transpose('k_left', 'b_left')
    .data
)

np_left_defect_op = (
    left_defect_op
    .transpose('k_left', 'b_left')
    .data
    .T
)

fp_list = [np_I, np_Z]
np_fp = multikron([
    fp_list[i%2]
    for i in range(domains_dict['num_defect_sites'])
])

left_defect_op_invariant = np.trace(
    np_fp
    @ np_left_defect_op.conj().T
    @ np_fp
    @ np_left_defect_op
    @ np_left_rdm
)

np_right_rdm = (
    right_rdm
    .transpose('k_right', 'b_right')
    .data
)

np_right_defect_op = (
    right_defect_op
    .transpose('k_right', 'b_right')
    .data
    .T
)

right_defect_op_invariant = np.trace(
    np_fp
    @ np_right_defect_op.conj().T
    @ np_fp
    @ np_right_defect_op
    @ np_right_rdm
)

np_sym_defect_m = get_symmetry_defect_m(domains_dict['num_defect_sites'])

left_cocyles = get_cocycles_from_rho(
    np_left_rdm,
    np_left_defect_op,
    np_sym_defect_m
)

right_cocyles = get_cocycles_from_rho(
    np_right_rdm,
    np_right_defect_op,
    np_sym_defect_m
)

all_cocyles = np.array([left_cocyles, right_cocyles])

cocycle_equation_output = compute_cocyle_equation_from_cocycles(all_cocyles)

cocycle_invariants = (all_cocyles[:,0,0]**2)*(all_cocyles[:,1,1])

In [264]:
all_cocyles[1].shape

(3, 3)

In [265]:
np.round(all_cocyles[1])

array([[ 1.+0.j,  0.-1.j, -1.+0.j],
       [-1.+0.j,  1.+0.j,  1.+0.j],
       [ 1.+0.j,  0.-1.j,  1.+0.j]])

In [256]:
right_cocyles = get_cocycles_from_rho(
    np_right_rdm,
    np_right_defect_op,
    np_sym_defect_m
)

In [266]:
defect_rho = np_right_rdm
defect_op = np_right_defect_op
unitary_sym_op = np_sym_defect_m

In [267]:
defect_op_list = get_all_defect_ops_from_t_defect_op(
    defect_op,
    unitary_sym_op
)

# Test - Cluster state

In [97]:
domains_dict = {
    'num_system_sites': 24,
    'left_projector_sites': list(range(4, 8)),
    'right_projector_sites': list(range(16, 20)),
    'num_projector_pad_sites': 2,
    'num_defect_sites': 2,
    'fdlu_depth': 0,
    'fdlu_offset': 0
}

In [98]:
cluster_psi = get_cluster_state_qu_tensor_network(domains_dict['num_system_sites'])

In [99]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:13<00:00,  1.51it/s]


### Analyze results

#### Projector scores

In [100]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([ 0.,  0.,  0.,  0., -0.,  0., -0., -0.,  0., -0.,  0., -0.,  0.,
        0.,  0.,  0., -0.,  0., -0.,  0.])

In [101]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

array([-0.,  0.,  0., -0.,  0., -0., -0., -0.,  0.,  0.,  0., -0., -0.,
        0., -0.,  0.,  0., -0.,  0.,  0.])

In [102]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [103]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

In [104]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0.])

#### Purities

In [105]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j])

In [106]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j])

In [107]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j])

Purities are the same, which one could likely prove.

In [108]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

array([1.+0.j, 1.-0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.-0.j, 1.+0.j, 1.-0.j, 1.+0.j, 1.-0.j, 1.-0.j, 1.+0.j, 1.+0.j,
       1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j])

#### Cut state results

In [109]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

array([ 1.-3.78640785e-18j, -1.+3.81772135e-18j, -1.+1.42758190e-17j,
        1.+1.03101589e-17j, -1.-1.50316029e-17j, -1.-2.79993761e-18j,
       -1.-1.66589314e-17j,  1.-2.06270653e-17j, -1.+1.51270839e-18j,
       -1.+1.93261934e-17j,  1.+8.55485492e-18j,  1.+1.28677590e-17j,
        1.-1.62473690e-17j, -1.-3.66893593e-18j, -1.-6.25862832e-18j,
        1.-7.38922396e-18j, -1.-3.18675889e-18j,  1.+9.93683909e-18j,
       -1.-4.98380590e-18j,  1.+5.22413751e-18j])

So the FP of the cut state can be even or odd. Interesting!

In [110]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

#### Defect op scores overlaps

In [111]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [112]:
np.array([d['defect_ops_scores'][-1] for d in results])

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1.])

In [113]:
[d['defect_ops_scores'][-1] for d in results]

[np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(0.9999999999999998),
 np.float64(0.9999999999999998),
 np.float64(0.9999999999999998),
 np.float64(0.9999999999999996),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000004),
 np.float64(1.0),
 np.float64(1.0000000000000004),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(0.9999999999999998)]

#### Phases

In [114]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [115]:
left_phases.shape

(20,)

In [116]:
np.round(left_phases, 3)

array([-1.-0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.-0.j,
       -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j,
       -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.+0.j])

In [117]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [118]:
right_phases.shape

(20,)

In [119]:
np.round(right_phases, 3)

array([-1.-0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.-0.j, -1.-0.j,
       -1.-0.j, -1.+0.j, -1.-0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.-0.j,
       -1.-0.j, -1.-0.j, -1.-0.j, -1.+0.j, -1.-0.j, -1.-0.j])

#### Cocyle information

In [120]:
np.array([
    d['cocycle_invariants'] for d in results
])

array([[ 1.38777878e-17-1.92593006e-34j,  6.93480707e-01+1.00321327e-16j],
       [ 6.91473864e-01+1.70664680e-16j, -9.78741860e-01+1.89162126e-16j],
       [-2.65149595e-02+2.81971384e-16j,  1.17399226e-01+8.87402988e-18j],
       [ 1.11053642e-01+3.63919196e-16j,  1.52660830e-16-3.09942969e-32j],
       [-1.11022302e-16-1.11022303e-16j,  3.91865123e-16+5.55111512e-17j],
       [ 9.90000107e-01-5.55112522e-17j,  4.96412589e-16-5.55111512e-17j],
       [-1.21430643e-16-1.34815155e-32j,  2.08892804e-16+4.72924622e-32j],
       [ 9.52583286e-01-5.95341586e-26j, -1.06428489e-01-2.07201915e-17j],
       [-2.01088089e-02-3.30834381e-16j,  9.94100403e-01-7.16324642e-16j],
       [-5.55111512e-17+5.55111512e-17j, -5.51120077e-17+5.55111512e-17j],
       [-1.66533454e-16-4.44089210e-16j,  1.39293402e-01-2.22777225e-17j],
       [ 1.28455527e-01+1.42614357e-17j, -1.79735830e-16+1.11022302e-16j],
       [-1.11022302e-16+3.33066907e-16j,  1.08723483e-16-3.88578059e-16j],
       [ 4.47213595e-01-2

In [133]:
X = results[0]['cocycle_equation_output']

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [134]:
X.shape

(2, 4, 4, 4)

In [135]:
np.round(X, 3)

array([[[[ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j],
         [ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00-0.00000000e+00j],
         [ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00-0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j],
         [ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00-0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j]],

        [[ 1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00+0.00000000e+00j,
           1.00000000e+00-0.00000000e+00j],
         [ 1.00000000e+00+0.00000000e+00j,
           3.34760030e+15+1.80568156e+16j,
           4.60000000e-02-2.51000000e-01j,
    

In [139]:
X[:, 0]

array([[[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]],

       [[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]]])

In [140]:
X[:, : ,0]

array([[[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]],

       [[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]]])

In [141]:
X[:, : , :, 0]

array([[[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.+0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]],

       [[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.-0.j],
        [1.+0.j, 1.-0.j, 1.+0.j, 1.+0.j],
        [1.+0.j, 1.+0.j, 1.-0.j, 1.+0.j]]])

In [142]:
sub_X = X[:, 1:, 1:, 1:]

In [146]:
sub_X.shape

(2, 3, 3, 3)

In [145]:
sub_X[0]

array([[[ 3.34760030e+15+1.80568156e+16j,
          4.64572866e-02-2.50588655e-01j,
         -1.00000000e+00+1.38777882e-17j],
        [-9.92602073e-18+5.35405396e-17j,
          2.80627948e-16-2.02460331e-16j,
         -9.92602073e-18+5.35405395e-17j],
        [-1.00000000e+00+4.27546122e-33j,
          8.19936485e+15+7.83227884e+15j,
          3.34760030e+15+1.80568156e+16j]],

       [[-7.20575940e+16-2.92601747e-08j,
          8.86475203e-01-1.35543360e+00j,
          1.00000000e+00-1.38777882e-17j],
        [-1.38777878e-17-1.92592994e-34j,
          1.00000000e+00+0.00000000e+00j,
         -1.38777878e-17-1.92592994e-34j],
        [ 1.00000000e+00-1.38777882e-17j,
          3.37957370e-01+5.16741780e-01j,
         -7.20575940e+16-2.93948689e-08j]],

       [[-8.19936485e+15+7.83227884e+15j,
         -3.34760030e+15+1.80568156e+16j,
         -1.00000000e+00+4.27546122e-33j],
        [ 6.37714751e-17+6.09164227e-17j,
          2.80627948e-16-2.02460331e-16j,
          6.37714751e-1

In [124]:
X = results[0]['cocycles']

In [125]:
X.shape

(2, 3, 3)

In [126]:
np.round(X[0], 3)

array([[ 1.-0.j, -0.+0.j, -1.-0.j],
       [-1.+0.j,  0.+0.j,  1.-0.j],
       [ 1.-0.j, -0.-0.j,  1.-0.j]])

In [127]:
np.round(X[1], 3)

array([[ 1.   +0.j  , -0.417-0.82j, -1.   -0.j  ],
       [-1.   -0.j  ,  0.693+0.j  ,  1.   +0.j  ],
       [ 1.   +0.j  , -0.417-0.82j,  1.   +0.j  ]])

In [128]:
X = results[1]['cocycles']

In [129]:
X.shape

(2, 3, 3)

In [130]:
np.round(X[0], 3)

array([[ 1.   +0.j   , -0.546-0.715j, -1.   +0.j   ],
       [-1.   -0.j   ,  0.691+0.j   ,  1.   +0.j   ],
       [ 1.   -0.j   , -0.546-0.715j,  1.   +0.j   ]])

In [131]:
np.round(X[1], 3)

array([[ 1.   -0.j   ,  0.087-0.055j, -1.   +0.j   ],
       [-1.   +0.j   , -0.979+0.j   ,  1.   -0.j   ],
       [ 1.   -0.j   ,  0.087-0.055j,  1.   -0.j   ]])

In [149]:
results = list()

for _ in tqdm(range(20)):
    current = find_invariants_via_projectors_from_random_state(
        cluster_psi,
        domains_dict,
        jw_even=True
    )
    results.append(current)

  0%|                                                                                                                                                                                     | 0/20 [00:00<?, ?it/s]


ValueError: The index b6 appears more than twice! If this is intentionally a 'hyper' tensor network you will need to explicitly supply `output_inds` when contracting for example.

### Analyze results

#### Projector scores

In [ ]:
np.round(np.array([
    d['left_proj_vec_results'][0][-1]
    for d in results
]), 3)

In [ ]:
np.round(np.array([
    d['right_proj_vec_results'][0][-1]
    for d in results
]), 3)

In [ ]:
def get_schmidt_vals_ratio(schmidt_vals):
    if len(schmidt_vals.data) > 1:
        return schmidt_vals.data[1]/schmidt_vals.data[0]
    else:
        return 0

In [ ]:
np.round(np.array([
    get_schmidt_vals_ratio(d['left_proj_vec_results'][1])
    for d in results
]), 3)

In [ ]:
np.round(np.array([
    get_schmidt_vals_ratio(d['right_proj_vec_results'][1])
    for d in results
]), 3)

#### Purities

In [ ]:
np.round(np.array(
    [d['cut_rho_purity'] for d in results]
), 3)

In [ ]:
np.round(np.array(
    [d['sub_cut_rho_trace'] for d in results]
), 3)

In [ ]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

Purities are the same, which one could likely prove.

In [ ]:
np.round(np.array(
    [d['sub_cut_rho_purity'] for d in results]
), 3)

#### Cut state results

In [ ]:
np.array(
    [d['cut_state_fermion_parity'] for d in results]
)

So the FP of the cut state can be even or odd. Interesting!

In [ ]:
np.round(np.array(
    [d['cut_overlap'] for d in results]
), 3)

#### Defect op scores overlaps

In [ ]:
np.round(np.array([d['defect_ops_scores'][-1] for d in results]), 5)

In [ ]:
np.array([d['defect_ops_scores'][-1] for d in results])

In [ ]:
[d['defect_ops_scores'][-1] for d in results]

#### Phases

In [ ]:
left_phases = np.array([
    d['left_defect_op_invariant'] for d in results
])

In [ ]:
left_phases.shape

In [ ]:
np.round(left_phases, 3)

In [ ]:
right_phases = np.array([
    d['right_defect_op_invariant'] for d in results
])

In [ ]:
right_phases.shape

In [ ]:
np.round(right_phases, 3)

#### Cocyle information

In [ ]:
np.array([
    d['cocycle_invariants'] for d in results
])

In [ ]:
X = results[0]['cocycle_equation_output']

In [ ]:
X.shape

In [ ]:
np.round(X, 3)

In [ ]:
X[:, 0]

In [ ]:
X[:, : ,0]

In [ ]:
X[:, : , :, 0]

In [ ]:
sub_X = X[:, 1:, 1:, 1:]

In [ ]:
sub_X.shape

In [ ]:
sub_X[0]

In [ ]:
X = results[0]['cocycles']

In [ ]:
X.shape

In [ ]:
np.round(X[0], 3)

In [ ]:
np.round(X[1], 3)

In [ ]:
X = results[1]['cocycles']

In [ ]:
X.shape

In [ ]:
np.round(X[0], 3)

In [ ]:
np.round(X[1], 3)